# 🚦 Train RT-DETR (Transformer) on Vadodara Traffic Dataset

This Google Colab notebook fine-tunes **RT-DETR (Real-Time DEtection TRansformer)** on your custom 9-class traffic dataset (`Bike`, `Bus`, `Car`, `Cart`, `Person`, `Rickshaw`, `Suzuki`, `Truck`, `Van`).

### ⚡ Step 0: Ensure GPU is Enabled
In the menu above, click **Runtime** > **Change runtime type** > Select **T4 GPU** > Click **Save**.

In [ ]:
# Step 1: Verify GPU & Install Dependencies
!nvidia-smi
!pip install -q ultralytics

### 📦 Step 2: Upload Your Dataset (`traffic_dataset_ready.zip`)
Run the cell below. It will open an upload button. Select **`traffic_dataset_ready.zip`** (located in your local `Downloads` folder, ~51 MB).

In [ ]:
from google.colab import files
import zipfile
import os

print("Please upload traffic_dataset_ready.zip:")
uploaded = files.upload()

zip_filename = [k for k in uploaded.keys() if k.endswith('.zip')][0]
print(f"Unzipping {zip_filename}...")

with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall('traffic_dataset_clean')

print("✅ Dataset extracted successfully!")
!ls -la traffic_dataset_clean

### 🧠 Step 3: Train RT-DETR (Baidu Real-Time Transformer)
We load the pretrained `rtdetr-l.pt` checkpoint and fine-tune for 30 epochs on your custom classes.

In [ ]:
from ultralytics import RTDETR

# 1. Load pretrained RT-DETR large model
model = RTDETR('rtdetr-l.pt')

# 2. Train on your 9-class traffic dataset
results = model.train(
    data='traffic_dataset_clean/data.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    name='rtdetr_traffic_vadodara',
    project='traffic_training',
    device=0
)
print("✅ Training Complete!")

### 📊 Step 4: Display Training Accuracy, F1-Score & Confusion Matrix
Run this cell to view your model's exact metrics, including the F1-Score curve, Confusion Matrix, and Loss curves.

In [ ]:
from IPython.display import Image, display
import glob
import os

possible_dirs = glob.glob('/content/runs/**/rtdetr_traffic_vadodara', recursive=True) + glob.glob('**/rtdetr_traffic_vadodara', recursive=True)
train_dir = [d for d in possible_dirs if os.path.exists(d)][0]
print(f"Viewing results from: {train_dir}")

print("\n--- 📈 1. TRAINING & VALIDATION LOSS / mAP CURVES ---")
if os.path.exists(f'{train_dir}/results.png'):
    display(Image(filename=f'{train_dir}/results.png', width=900))

print("\n--- 🎯 2. F1-SCORE CURVE (Confidence vs F1) ---")
for f1_file in ['F1_curve.png', 'BoxF1_curve.png']:
    if os.path.exists(f'{train_dir}/{f1_file}'):
        display(Image(filename=f'{train_dir}/{f1_file}', width=700))
        break

print("\n--- 🔀 3. CONFUSION MATRIX (Per-class performance) ---")
if os.path.exists(f'{train_dir}/confusion_matrix.png'):
    display(Image(filename=f'{train_dir}/confusion_matrix.png', width=700))

print("\n--- 📉 4. PRECISION-RECALL CURVE ---")
for pr_file in ['PR_curve.png', 'BoxPR_curve.png']:
    if os.path.exists(f'{train_dir}/{pr_file}'):
        display(Image(filename=f'{train_dir}/{pr_file}', width=700))
        break

### 💾 Step 5: Download Trained Model (`best.pt`)
Download your fine-tuned weights to use directly in your Vadodara Smart City project.

In [ ]:
from google.colab import files
import glob
import os

candidates = glob.glob('/content/runs/**/rtdetr_traffic_vadodara/weights/best.pt', recursive=True) + glob.glob('**/weights/best.pt', recursive=True)
if candidates:
    best_weights = candidates[0]
    print(f"Downloading {best_weights} to your computer...")
    files.download(best_weights)
else:
    print("Weights file not found.")